In [8]:
import pandas as pd
from vae_module import train_vae, encode, decode, load_vae
import numpy as np

In [9]:
wti_df = pd.read_csv('../data/wti_prices.csv', index_col=0, parse_dates=['Date'])
wti_df = np.log(wti_df).diff().dropna()
cols = list(wti_df.columns)
print(wti_df.shape)
print(cols)

(480, 1)
['WTI_price']


In [10]:
predictors = pd.read_csv('../data/predictor_data.csv', index_col=0, parse_dates=['Date'])
cols = list(predictors.columns)
print(predictors.shape)
print(cols)

(481, 8)
['CPI', 'TB3MS', 'M1', 'M2', 'AUD_USD', 'CAD_USD', 'NZD_USD', 'ZAR_USD']


In [12]:
# Combine TB3MS and WTI data by month (index)
combined_df = predictors.join(wti_df, how='inner').sort_index()

# Optional: set a clear index name
combined_df.index.name = "month"

print(combined_df.shape)
display(combined_df.tail())
display(combined_df.isna().sum())

# can see that CPI and last month of currency got issues, so we ffill those with the previous month's value
combined_df.fillna(method='ffill', inplace=True)
display(combined_df.isna().sum())

(480, 9)


,CPI,TB3MS,M1,M2,AUD_USD,CAD_USD,NZD_USD,ZAR_USD,WTI_price
month,,,,,,,,,
2025-09-01,324.245,3.92,18903.9,22169.8,0.659757,1.383429,0.588419,17.425195,-0.013973
2025-10-01,NaN,3.82,18986.0,22250.4,0.654682,1.398768,0.576491,17.272164,-0.049189
2025-11-01,325.063,3.78,19022.7,22296.5,0.650172,1.405617,0.565206,17.229661,-0.013725
2025-12-01,326.031,3.59,19100.8,22386.9,0.664264,1.379491,0.578691,16.832650,-0.035418
2026-01-01,326.588,3.57,19201.1,22469.1,NaN,NaN,NaN,NaN,0.035085


CPI          1
TB3MS        0
M1           0
M2           0
AUD_USD      1
CAD_USD      1
NZD_USD      1
ZAR_USD      1
WTI_price    0
dtype: int64

C:\Users\Josiah Lee\AppData\Local\Temp\ipykernel_19084\2346409964.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  combined_df.fillna(method='ffill', inplace=True)


CPI          0
TB3MS        0
M1           0
M2           0
AUD_USD      0
CAD_USD      0
NZD_USD      0
ZAR_USD      0
WTI_price    0
dtype: int64

In [6]:
combined_df.isna().sum()

CPI          1
TB3MS        0
M1           0
M2           0
AUD_USD      1
CAD_USD      1
NZD_USD      1
ZAR_USD      1
WTI_price    0
dtype: int64

In [16]:
from data_pipeline import build_pipeline, inverse_transform, load_scaler

# Build windowed dataset from raw training DataFrame
windows, scaler = build_pipeline(
    df=combined_df,            # rows = timesteps, columns = variables
    scaler_save_path="scaler.pkl"
)
# windows shape: (num_windows, window_length, num_variables)

a = windows.copy()

[Pipeline] Observations: 480 | Variables: 2
[Pipeline] Columns: ['TB3MS', 'WTI_price']
[Pipeline] Window length: 24 | Stride: 1
[Pipeline] Windows to be created: 457
[Pipeline] Scaler: minmax
------------------------------------------------------------
[Pipeline] Normalisation complete.
[Pipeline] Data range after scaling: [0.0000, 1.0000]
[Pipeline] Windows created: 457
[Pipeline] Output shape: (457, 24, 2)  (num_windows, window_length, num_variables)
[Pipeline] Scaler saved to: scaler.pkl
------------------------------------------------------------
[Pipeline] Pipeline complete. Ready for Module 2 (VAE).


In [17]:
windows.shape

(457, 24, 2)

In [18]:
# Train
from vae_module import VAEConfig

config = VAEConfig(
    num_epochs=300,
    latent_dim=1,             # force real compression
    kl_warmup_epochs=5,
    kl_weight_max=2.0,
    learning_rate=1e-4,
    verbose_every=25,
)

model, history = train_vae(
    data=windows,        # shape: (num_windows, window_len, num_variables)
    config=config,
    save_path="vae_checkpoint.pt"
)

# Encode new data
latent_sequences = encode(windows, model)

# Decode latent sequences back to data space
reconstructed = decode(latent_sequences, model)

# Load a saved model
model = load_vae("vae_checkpoint.pt")


[VAE] Device: cuda
[VAE] Input dim: 2 | Latent dim: 1 | Hidden dim: 16
[VAE] Windows: 457 | Window length: 24 | Variables: 2
[VAE] Train windows: 389 | Val windows: 68
[VAE] KL warmup: 5 epochs | Max KL weight: 2.0
------------------------------------------------------------
[VAE] Epoch    1/300 | KL weight: 0.4000 | Train — Total: 0.261916, Recon: 0.247976, KL: 0.034849 | Val   — Total: 0.255701, Recon: 0.242283, KL: 0.033543
[VAE] Epoch   25/300 | KL weight: 2.0000 | Train — Total: 0.114124, Recon: 0.112188, KL: 0.000968 | Val   — Total: 0.111912, Recon: 0.109907, KL: 0.001002
[VAE] Epoch   50/300 | KL weight: 2.0000 | Train — Total: 0.047255, Recon: 0.047064, KL: 0.000096 | Val   — Total: 0.051198, Recon: 0.050757, KL: 0.000221
[VAE] Epoch   75/300 | KL weight: 2.0000 | Train — Total: 0.024364, Recon: 0.024222, KL: 0.000071 | Val   — Total: 0.028816, Recon: 0.028453, KL: 0.000182
[VAE] Epoch  100/300 | KL weight: 2.0000 | Train — Total: 0.015801, Recon: 0.015683, KL: 0.000059 | Val 

In [19]:
import numpy as np

latent = encode(windows, model)
latent_flat = latent.reshape(-1, latent.shape[-1])

print("Latent mean per dimension:", latent_flat.mean(axis=0).round(4))
print("Latent std per dimension: ", latent_flat.std(axis=0).round(4))
print("Min std:", latent_flat.std(axis=0).min().round(6))

Latent mean per dimension: [-0.0001]
Latent std per dimension:  [0.005]
Min std: 0.005047


In [24]:
# Train
from timegan_module import train_timegan, generate, load_timegan, TimeGANConfig
import torch

config = TimeGANConfig(
    hidden_dim=32,
    num_layers=2,              # increase depth slightly
    noise_dim=2,
    embedding_dim=32,
    num_epochs=2000,           # more total epochs
    batch_size=32,
    learning_rate=5e-4,        # reduce from 1e-3 — more stable
    gamma=1.0,
    phase_split=(0.1, 0.3, 0.6),  # more epochs to supervised phase
    early_stopping_patience=5,
    eval_every_n_epochs=100,   # evaluate less frequently
    verbose_every=100,
    random_seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

model, history = train_timegan(
    data=a,               # shape: (num_windows, window_length, num_variables)
    save_path="timegan_checkpoint.pt"
)

# Generate synthetic windows
synthetic = generate(n_samples=200, model=model)
# synthetic shape: (200, window_length, num_variables)

# Load saved model
model = load_timegan("timegan_checkpoint.pt")

[TimeGAN] Device: cuda
[TimeGAN] Input dim: 2 | Embedding dim: 32 | Hidden dim: 32 | Noise dim: 2
[TimeGAN] Windows: 457 | Seq length: 24
[TimeGAN] Total epochs: 1000 | Phase split — Embed: 200 | Supervised: 200 | Joint: 600
[TimeGAN] Early stopping: enabled (patience=5)

[Phase 1/3] Embedding Pretraining (200 epochs)
----------------------------------------------------------------------
  [Embedding] Epoch    1/200 | Loss: 0.113893
  [Embedding] Epoch   50/200 | Loss: 0.002172
  [Embedding] Epoch  100/200 | Loss: 0.001142
  [Embedding] Epoch  150/200 | Loss: 0.000707
  [Embedding] Epoch  200/200 | Loss: 0.000579

[Phase 2/3] Supervised Pretraining (200 epochs)
----------------------------------------------------------------------
  [Supervised] Epoch    1/200 | Loss: 0.001450
  [Supervised] Epoch   50/200 | Loss: 0.001415
  [Supervised] Epoch  100/200 | Loss: 0.001414
  [Supervised] Epoch  150/200 | Loss: 0.001421
  [Supervised] Epoch  200/200 | Loss: 0.001399

[Phase 3/3] Joint GAN T

In [25]:
import numpy as np

for v in range(a.shape[2]):
    flat = a[:, :, v].flatten()
    print(f"Variable {v}: mean={flat.mean():.4f} | std={flat.std():.4f} | "
          f"min={flat.min():.4f} | max={flat.max():.4f}")

Variable 0: mean=0.6228 | std=0.0979 | min=0.0000 | max=1.0000
Variable 1: mean=0.5129 | std=0.0847 | min=0.0000 | max=1.0000


In [27]:
for v in range(a.shape[2]):
    real_v = a[:, :, v].flatten()
    fake_v = synthetic[:, :, v].flatten()
    print(f"Variable {v}:")
    print(f"  Real  — mean: {real_v.mean():.4f} | std: {real_v.std():.4f}")
    print(f"  Synth — mean: {fake_v.mean():.4f} | std: {fake_v.std():.4f}")

Variable 0:
  Real  — mean: 0.6228 | std: 0.0979
  Synth — mean: 0.6212 | std: 0.0841
Variable 1:
  Real  — mean: 0.5129 | std: 0.0847
  Synth — mean: 0.5113 | std: 0.0304


In [28]:
real_flat = a.reshape(-1, a.shape[2])
fake_flat = synthetic.reshape(-1, synthetic.shape[2])

real_corr = np.corrcoef(real_flat.T)
fake_corr = np.corrcoef(fake_flat.T)

print("Real correlation matrix:\n", np.round(real_corr, 4))
print("Synthetic correlation matrix:\n", np.round(fake_corr, 4))
print(f"Frobenius norm: {np.linalg.norm(real_corr - fake_corr, 'fro'):.4f}")

Real correlation matrix:
 [[1.     0.1611]
 [0.1611 1.    ]]
Synthetic correlation matrix:
 [[1.     0.9753]
 [0.9753 1.    ]]
Frobenius norm: 1.1515


In [29]:
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt

n_lags = 20
synthetic = generate(len(windows), model)

fig, axes = plt.subplots(1, windows.shape[2], figsize=(6 * windows.shape[2], 4))
if windows.shape[2] == 1:
    axes = [axes]

for v in range(windows.shape[2]):
    real_series = windows[:, :, v].flatten()
    fake_series = synthetic[:, :, v].flatten()
    
    real_acf = acf(real_series, nlags=n_lags, fft=True)
    fake_acf = acf(fake_series, nlags=n_lags, fft=True)
    
    axes[v].plot(real_acf, label='Real', marker='o', markersize=4)
    axes[v].plot(fake_acf, label='Synthetic', marker='s', markersize=4)
    axes[v].axhline(0, color='black', linewidth=0.5)
    axes[v].set_title(f'ACF Comparison — Variable {v}')
    axes[v].set_xlabel('Lag')
    axes[v].legend()
    axes[v].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("acf_comparison.png", dpi=150)

In [26]:
import matplotlib.pyplot as plt
import numpy as np

synthetic = generate(len(a), model)

fig, axes = plt.subplots(3, 2, figsize=(14, 8))
var_names = ['var_1', 'var_2']  # replace with your actual column names

for i, ax_row in enumerate(axes):
    idx = i * 30
    ax_row[0].plot(a[idx])
    ax_row[0].set_title(f'Real Window {idx}')
    ax_row[0].legend(var_names)
    
    ax_row[1].plot(synthetic[idx])
    ax_row[1].set_title(f'Synthetic Window {idx}')
    ax_row[1].legend(var_names)

plt.tight_layout()
plt.savefig("visual_overlay.png", dpi=150)

In [ ]:
# Step 1 — Module 1
windows, scaler = build_pipeline(df, scaler_save_path="scaler.pkl")

# Step 2 — Module 2 (skip for Option B, enable for hybrid)
# model_vae, _ = train_vae(windows, save_path="vae_checkpoint.pt")
# latent = encode(windows, model_vae)

# Step 3 — Module 3 (pass windows for Option B, latent for hybrid)
model_gan, history = train_timegan(windows, save_path="timegan_checkpoint.pt")

# Step 4 — Generate
synthetic_normalised = generate(500, model_gan)
synthetic_original = inverse_transform(synthetic_normalised, scaler)

In [ ]:
from evaluation_module import EvaluationConfig, evaluate_all

config = EvaluationConfig(
    n_lags=12,
    random_seed=42,
    report_save_path="results_timegan.txt",
    json_save_path="results_timegan.json",
    figure_save_path="figure_timegan.png",
)

# Run identically for each generator
results_jitter  = evaluate_all(real, synthetic_jitter,  config, variable_names=["TB3MS", "WTI_price"])
results_timegan = evaluate_all(real, synthetic_timegan, config, variable_names=["TB3MS", "WTI_price"])
results_hybrid  = evaluate_all(real, synthetic_hybrid,  config, variable_names=["TB3MS", "WTI_price"])